# Preguntas

- **¿Cómo cambió el sonido de la música a lo largo de las décadas?**

In [20]:
# Importamos librerias
import pandas as pd
import plotly.express as px

## Cargar archivo limpio

In [21]:
df_final = pd.read_csv("../data/spotify_limpio.csv")
df_final.shape

(165424, 14)

## Resumir por decadas

In [22]:
caracteristicas = ["energy", "acousticness", "danceability", "valence"]

por_decada = df_final.groupby("decada")[caracteristicas].median()
por_decada

,energy,acousticness,danceability,valence
decada,,,,
1920,0.2160,0.991,0.569,0.612
1930,0.2600,0.976,0.529,0.594
1940,0.2300,0.975,0.450,0.486
1950,0.2550,0.912,0.483,0.480
1960,0.3950,0.702,0.505,0.567
1970,0.5335,0.359,0.528,0.607
1980,0.6225,0.167,0.562,0.589
1990,0.6030,0.184,0.581,0.558
2000,0.6910,0.133,0.582,0.536


## Graficos

In [23]:


px.line(
    por_decada,
    markers=True,
    title="Cómo cambió el sonido por década",
    labels={"value": "Valor mediano (0 a 1)", "decada": "Década", "variable": "Característica"},
)

Entre 1960 y 1980 las canciones se volvieron mucho menos acústicas (la acústica mediana pasó de 0,91 a 0,17) y desde entonces se mantiene baja. La energía subió de forma sostenida desde los 60 y alcanzó su máximo en los 2000. La positividad (valence) bajó en las últimas décadas.

----

- **¿Qué tienen en común las canciones más populares? ¿Hay un “sonido” que haga que una canción sea más escuchada?**

In [24]:
# Canciones con popularidad real
con_pop = df_final[~df_final["pop_cero"]]
con_pop.shape

(141128, 14)

In [25]:
# Popularidad por decada 
popularidad_decada = con_pop.groupby("decada")["popularity"].median()
popularidad_decada

decada
1920     5.0
1930     4.0
1940     4.0
1950    12.0
1960    24.0
1970    33.0
1980    36.0
1990    43.0
2000    48.0
2010    59.0
2020    71.0
Name: popularity, dtype: float64

## Grafico

In [26]:
px.bar(
    popularidad_decada,
    title="Popularidad mediana por década",
    labels={"value": "Popularidad mediana", "decada": "Década"},
)

¿Que caracteristicas se relacionan con la popularidad?

In [27]:
cols = ["energy", "danceability", "valence", "acousticness", "loudness", "tempo", "duracion_min"]

con_pop[cols + ["popularity"]].corr()["popularity"]

energy          0.373310
danceability    0.220650
valence        -0.012643
acousticness   -0.469760
loudness        0.396331
tempo           0.079845
duracion_min    0.053846
popularity      1.000000
Name: popularity, dtype: float64

In [28]:
# Canciones de 2000
solo_2000 = con_pop[con_pop["decada"] == 2000]

solo_2000[cols + ["popularity"]].corr()["popularity"]

energy          0.060659
danceability    0.061662
valence         0.028896
acousticness   -0.056068
loudness        0.092372
tempo           0.009878
duracion_min    0.009516
popularity      1.000000
Name: popularity, dtype: float64

---

- **¿En qué se diferencian los géneros entre sí, tanto por cuántas canciones tienen como por cómo suenan?**

In [29]:
# Canciones que tienen una familia real
con_genero = df_final[~df_final["familia"].isin(["Sin género", "Otros"])]
con_genero.shape

(143468, 14)

In [30]:
# Cuantas canciones hay en cada familia
cantidad = con_genero["familia"].value_counts()

px.bar(
    cantidad,
    title="Canciones por familia de género",
    labels={"value": "Cantidad de canciones", "familia": "Familia"},
)

In [31]:
# Como suena cada familia
perfil = con_genero.groupby("familia")[["energy", "danceability", "valence", "acousticness"]].median()
perfil.round(2)

,energy,danceability,valence,acousticness
familia,,,,
Clásica,0.15,0.32,0.18,0.97
Country / folk,0.44,0.58,0.56,0.56
Electrónica,0.70,0.64,0.52,0.08
Hip hop / rap,0.67,0.73,0.54,0.09
Jazz / blues,0.30,0.54,0.55,0.85
Latina,0.49,0.64,0.76,0.58
Metal / punk,0.90,0.44,0.43,0.00
Mundo / regional,0.32,0.51,0.62,0.96
Pop,0.38,0.53,0.48,0.71


In [32]:
px.imshow(perfil, text_auto=".2f", aspect="auto", title="Perfil sonoro por familia de género")

In [33]:
# ¿Que familia es mas popular?
recientes = con_genero[(~con_genero["pop_cero"]) & (con_genero["year"] >= 1990)]

recientes.groupby("familia")["popularity"].median().sort_values(ascending=False)

familia
Pop                  57.0
Hip hop / rap        55.0
Electrónica          55.0
Latina               51.0
R&B / soul / funk    49.0
Country / folk       49.0
Rock                 49.0
Metal / punk         48.0
Reggae / caribeña    48.0
Clásica              47.0
Jazz / blues         45.0
Mundo / regional     44.0
Name: popularity, dtype: float64

---

- **¿Quiénes lideran en el dataset y cómo se comparan entre sí?**

In [36]:
# Ranking por cantidad de canciones
top_cantidad = df_final["artista_principal"].value_counts().head(15)

px.bar(
    top_cantidad.sort_values(),
    orientation="h",
    title="Artistas con más canciones en el dataset",
    labels={"value": "Cantidad de canciones", "artista_principal": "Artista"},
)

In [ ]:
# Ranking por popularidad
con_pop = df_final[~df_final["pop_cero"]]

por_artista = con_pop.groupby("artista_principal").agg(
    canciones=("popularity", "size"),
    popularidad=("popularity", "median"),
)

frecuentes = por_artista[por_artista["canciones"] >= 30]

frecuentes.sort_values("popularidad", ascending=False).head(15)

,canciones,popularidad
artista_principal,,
Billie Eilish,43,76.0
Juice WRLD,69,75.0
Bad Bunny,58,74.0
Joji,44,73.0
Anuel AA,30,72.5
Pop Smoke,32,72.5
The Kid LAROI,34,72.0
Post Malone,55,71.0
Lil Peep,50,70.5


In [35]:
# Comparar 2 artistas
df_final[df_final["artista_principal"].str.contains("Queen")]["artista_principal"].unique()

array(['Queen', 'Queensrÿche', 'Queens of the Stone Age', 'Queen Latifah',
       'Ivy Queen', 'Queen Naija', 'Queen Pen', 'Queen Key',
       'Queen Sarah Saturday', 'Christine and the Queens'], dtype=object)

In [ ]:
elegidos = ["Queen", "The Beatles"]   # cambialos por los tuyos

comparacion = (
    df_final[df_final["artista_principal"].isin(elegidos)]
    .groupby("artista_principal")[["energy", "danceability", "valence", "acousticness"]]
    .median()
)

px.bar(comparacion, barmode="group", title="Comparación de dos artistas")

---

# Conclusiones

## Pregunta central
**¿Cómo cambió la música a lo largo de las décadas y qué hace que una canción sea más popular que otra?**

## Respuesta breve
El sonido de las canciones cambió muchísimo entre 1960 y 1980 y después se estabilizó. En cambio, la popularidad depende mucho más de la **época** y del **artista** que de las características de audio de cada canción.

---

## 1. Evolución: un gran cambio entre los 60 y los 80
- **La música se volvió mucho menos acústica.** La acústica mediana pasó de 0,91 en los 50 a 0,70 en los 60, 0,36 en los 70 y 0,17 en los 80. Desde entonces se mantiene baja (entre 0,13 y 0,18).
- **La energía subió de forma sostenida**: de 0,26 en los 50 a un máximo de 0,69 en los 2000, con una leve baja después (0,66 y 0,64).
- **El volumen aumentó** de unos -14 dB en los 50 a unos -6 dB en los 2000.
- **La positividad (`valence`) bajó**: de 0,61 en los 70 a 0,44 en los 2010.
- **La duración mediana creció hasta los 80** (de unos 3 minutos en los 60 a unos 4 minutos) y desde entonces fue bajando (3,6 minutos en los 2010).

*Lectura:* las canciones del catálogo son hoy menos acústicas, más enérgicas y más fuertes que las de mediados del siglo XX. La causa no se puede determinar con estos datos.

## 2. Popularidad: pesa más la época que el sonido
- **La popularidad crece década tras década.** La mediana pasa de unos 4 puntos en los años 30 y 40 a 12 en los 50, 33 en los 70, 48 en los 2000 y 59 en los 2010. Como la métrica es la **actual**, favorece a la música reciente.
- **Con todas las décadas juntas hay correlaciones moderadas**: la acústica se asocia negativamente con la popularidad (aprox. -0,47) y la energía y el volumen positivamente (aprox. 0,37 y 0,40).
- **Dentro de una misma década, esa relación casi desaparece**: ninguna característica supera aproximadamente 0,2 en valor absoluto.

*Lectura:* el vínculo aparente entre popularidad y sonido se debe en gran parte a que las canciones recientes son a la vez más enérgicas y más escuchadas. Dentro de una misma época, las características de audio explican muy poco por qué una canción es más popular que otra. No hay una "fórmula sonora" del éxito, al menos con estas variables.

## 3. Géneros: familias con perfiles sonoros muy distintos
- **Rock es por lejos la familia más grande** (más de 36.000 canciones), seguida de Pop, Clásica, Jazz / blues y Latina. Electrónica, Mundo / regional y Reggae / caribeña son las más chicas.
- **Los perfiles sonoros son claros:**
  - Clásica es la más acústica (aprox. 0,97) y la menos enérgica (aprox. 0,15).
  - Metal / punk es la más enérgica (aprox. 0,90) y casi nada acústica.
  - Hip hop / rap y Reggae / caribeña son las más bailables (aprox. 0,73 y 0,75).
  - Reggae / caribeña y Latina son las más "positivas" (valence aprox. 0,78 y 0,76).
- **En popularidad (canciones desde 1990) las diferencias son moderadas**: la mediana va de unos 44 puntos (Mundo / regional) a unos 57 (Pop), seguida de cerca por Hip hop / rap y Electrónica.

## 4. Artistas: el ranking cambia según qué se mida
- **Por cantidad de canciones** lideran artistas con muchísimas grabaciones: Francisco Canaro (más de 2.200), Chopin, Mozart, Bach y Beethoven. Eso refleja cómo está armado el catálogo y no quién es más famoso.
- **Por popularidad** (artistas con 30 o más canciones) lideran artistas recientes, como Billie Eilish, Juice WRLD y Bad Bunny, y quedan al final artistas de los años 30 y 40. Es el mismo efecto de la época que aparece en la sección anterior.
- **El catálogo está muy repartido**: hay más de 20.000 artistas distintos y el 44% aparece con una sola canción. Los 100 artistas con más canciones reúnen solo cerca del 19% del total.

---

## Limitaciones
- La **popularidad es la actual**, no la de la época de lanzamiento.
- El año **2020 está incompleto** y los años 20 y 30 tienen pocas canciones, por lo que sus valores son menos firmes.
- El dataset es un **catálogo de Spotify**, no un registro de lo que la gente escuchaba en cada época.
- Las **familias de género** son una agrupación propia y aproximada, hecha con palabras clave y asignada por artista.
- Las canciones **sin género** (cerca del 8%) se concentran antes de 1950 y en artistas no anglosajones, por lo que esa sección subrepresenta esa música.
- Se usa el **primer artista** de cada canción, así que las colaboraciones se atribuyen a uno solo.
- La **correlación no implica causalidad**, y no se midieron factores que seguramente pesan mucho en el éxito (promoción, playlists, el propio artista).